# Supported-DPSS RFI flagger — marjum-2026-07

The **canonical RFI review interface** for this campaign. It is a thin viewer over
`eigsep_data.rfi_supported`: it selects data from `marjum-2026-07/data/`, applies
package defaults plus explicit overrides, runs the flagger, and presents the
aligned waterfall and band-summary views. It caches nothing — every input is read
from the campaign directory or the installed package at run time, so pointing it
at a different file range is the only edit needed.

Superseded RFI notebooks (`rfi_dev_v0/v1/v2`, `rfi_explorer`, `rfi_flag_prototype`)
moved to `debug/` on 2026-09-21; the delay-transform exploration that used to sit
at the end of this notebook is now `debug/rfi_dev_v2.1_delay_scratch.ipynb`.

**Status:** `v3-beta`; the method and product schemas remain subject to review.
Requires `eigsep_data` commit `9536969` or later, imported from the installed
package without modifying `sys.path`.


In [ ]:
from pathlib import Path
import os
from dataclasses import asdict, replace
import hashlib
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from IPython.display import display

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'marjum-2026-07/data').is_dir()
            and (p / 'marjum-2026-07/curation/antenna_resolution.json').is_file())

import eigsep_data
try:
    from eigsep_data import AntennaResolutionPolicy, MetadataIndex
    from eigsep_data.rfi_supported import (
        FLAG_BITS, REPORT_BANDS, RFIConfig, algorithm_source_sha256,
        run_selection,
    )
except (ImportError, ModuleNotFoundError) as exc:
    raise ImportError(
        'rfi_dev_v2.1 requires eigsep_data commit 9536969 or later; '
        f'imported package: {Path(eigsep_data.__file__).resolve()}'
    ) from exc

CAMPAIGN = ROOT / 'marjum-2026-07'
POLICY_PATH = CAMPAIGN / 'curation/antenna_resolution.json'
resolution_policy = AntennaResolutionPolicy.load(POLICY_PATH)
print('Campaign:', CAMPAIGN)
print('eigsep-data:', version('eigsep-data'))
print('Package:', Path(eigsep_data.__file__).resolve())
print('Algorithm: flags@v3-beta.1 / smooth_model@v3-beta.1')
print('Algorithm source sha256:', algorithm_source_sha256())
print('Antenna policy:', resolution_policy.provenance())

# ---------------------------------------------------------------- talk style
# One place to set figure appearance. TALK=True gives slide-legible type and
# line weights; TALK=False returns to compact notebook defaults. Figures are
# written to FIGDIR as PNG (raster, for slides) and PDF (vector, for print)
# whenever SAVE_FIGURES is set.
TALK = True
SAVE_FIGURES = False
FIGDIR = Path('figures')

plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 220,
    'savefig.bbox': 'tight',
    'font.size': 15 if TALK else 10,
    'axes.titlesize': 17 if TALK else 11,
    'axes.labelsize': 15 if TALK else 10,
    'axes.titleweight': 'semibold',
    'axes.labelpad': 6,
    'xtick.labelsize': 13 if TALK else 9,
    'ytick.labelsize': 13 if TALK else 9,
    'legend.fontsize': 13 if TALK else 9,
    'legend.frameon': False,
    'lines.linewidth': 2.2 if TALK else 1.2,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'image.interpolation': 'nearest',
    'figure.facecolor': 'white',
})


def save(fig, name):
    """Write `fig` to FIGDIR/name.{png,pdf} when SAVE_FIGURES is set."""
    if not SAVE_FIGURES:
        return
    FIGDIR.mkdir(exist_ok=True)
    for ext in ('png', 'pdf'):
        fig.savefig(FIGDIR / f'{name}.{ext}')
    print('wrote', FIGDIR / f'{name}.png')

In [ ]:
#import glob
#sorted(glob.glob('corr_20260717_195611Z.h5'))

## 1. Select input data

Set `FILES` to an inclusive pair of complete filenames, or set `TIME` to a `(start, end)` UTC/Unix pair. With both left as `None`, the default selects a 20-minute window at `OFFSET_S` after the start of `SPAN`. Physical antenna names are resolved by the campaign policy; stale `header/input_to_ant` values are not used.

In [ ]:
FILES = None
TIME = None

SPAN = 14
OFFSET_S = 16 * 3600 + 1800
MINUTES = 60

AIR_ANTENNA = 'box-air'
GROUND_ANTENNA = 'box-gnd'

idx = MetadataIndex(CAMPAIGN / 'data')
if FILES is not None:
    selection = idx.select(files=tuple(FILES))
elif TIME is not None:
    selection = idx.select(time=tuple(TIME))
else:
    all_meta = idx.select().meta.assign(visit=idx.select().visits(gap_s=600))
    visit = all_meta[all_meta.visit == SPAN]
    start = float(visit.time_best.min()) + OFFSET_S
    selection = idx.select(time=(start, start + MINUTES * 60))

approved_files, rejected, resolution_rules = resolution_policy.partition(selection.meta)
if rejected:
    display(pd.DataFrame(rejected))
    raise ValueError('selection contains files rejected by the antenna policy')
display(pd.Series(resolution_rules, name='antenna-resolution rule'))
display(selection.meta.groupby(['file', 'rfswitch'], dropna=False).size().rename('rows'))
print(selection.summary())

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

display(pd.Series({name: sha256(CAMPAIGN / 'data' / name)
                   for name in selection.files}, name='input sha256'))
print('Selected files:', len(approved_files))

## 2. Parameters

Edit only `OVERRIDES`. Empty overrides use the package defaults recorded in the product manifest. The background combines the full-band 50 ns by 1 mHz DPSS continuum basis with a 300 ns basis over 35–88 MHz whose amplitudes are constant over this window. The low-band correction is tapered over 5 MHz at both edges and fitted jointly with the continuum. Set `spectral_correction_halfwidth_s` to zero for the previous model.

Units are seconds for frequency half-widths, Hz for `time_halfwidth_hz`, MHz for frequency bands and taper width, and integrations/channels for guards.

In [ ]:
OVERRIDES = {
    # 'cross_cut': 8.0,
    # 'support_inflation': 5.0,
}

config = replace(RFIConfig(), **OVERRIDES)
display(pd.Series(asdict(config), name='v3-beta parameters'))

## 3. Generate the mask and supported model

Non-`RFANT` rows are excluded before fitting and receive bit 0 over the full band. The original continuum and constant-time low-band correction are solved together, and the design-support calculation covers their combined coefficient set. The public model remains NaN where those support checks fail; bit 7 records those cells. The in-memory `model_raw` is retained only for diagnosis.

In [ ]:
result = run_selection(
    selection,
    air_antenna=AIR_ANTENNA,
    ground_antenna=GROUND_ANTENNA,
    config=config,
    resolution_policy=resolution_policy,
)

air = selection.load_bundle(antenna=AIR_ANTENNA, band_mhz=config.band_mhz,
                            missing='raise', resolution_policy=resolution_policy)
residual = air.data - result.model
flagged_residual = np.where(result.mask, np.nan, residual)

display(pd.DataFrame(result.diagnostics['resolved_inputs']).T.rename_axis('file'))

summary = pd.Series({
    'rows': len(result.times),
    'channels': len(result.freqs_mhz),
    'excluded_fraction': result.mask.mean(),
    'rfi_evidence_fraction': result.rfi_mask.mean(),
    'supported_model_fraction': result.support_ok.mean(),
    'fit_training_fraction': result.fit_keep.mean(),
}, name='result')
display(summary)
segment_rows = []
for number, segment in enumerate(result.diagnostics['segments']):
    solver_history = segment.get('solver', [])
    last_solve = solver_history[-1] if solver_history else {}
    segment_rows.append({
        'segment': number,
        'status': segment.get('skipped', 'fit'),
        'samples': segment.get('samples'),
        'coefficients': segment.get('coefficients', last_solve.get('coefficients')),
        'stationary_frequency_modes': last_solve.get('stationary_frequency_modes'),
        'last_cg_iterations': last_solve.get('iterations'),
        'last_relative_normal_residual': last_solve.get('relative_normal_residual'),
    })
display(pd.DataFrame(segment_rows).set_index('segment'))
display(pd.Series({name: mask.mean() for name, mask in result.reasons.items()},
                  name='overlapping reason fractions'))
display(pd.DataFrame(FLAG_BITS).T)

band_rows = []
for name, (lo, hi) in {'35-50': (35, 50), '50-70': (50, 70), **REPORT_BANDS}.items():
    channel = (result.freqs_mhz >= lo) & (result.freqs_mhz < hi)
    band_rows.append({
        'band_mhz': name,
        'supported': result.support_ok[:, channel].mean(),
        'fit_training': result.fit_keep[:, channel].mean(),
        'retained': (~result.mask[:, channel]).mean(),
        'rfi_evidence': result.rfi_mask[:, channel].mean(),
    })
display(pd.DataFrame(band_rows).set_index('band_mhz'))

## 4. Shared data/model/flag/residual view

All five panels share frequency and time axes and remain interactive under `%matplotlib widget`. Residual panels use raw correlator counts. White gaps in the model are deliberately unsupported, and white gaps in the last panel are all excluded cells.

In [ ]:
freqs = result.freqs_mhz
minutes = (result.times - result.times[0]) / 60
extent = [freqs[0], freqs[-1], minutes[-1], 0]

positive = air.data[np.isfinite(air.data) & (air.data > 0)]
vmin, vmax = np.quantile(positive, [0.01, 0.99])
finite_residual = residual[np.isfinite(residual)]
rlim = np.quantile(np.abs(finite_residual), 0.9)
linthresh = max(1.0, np.quantile(np.abs(finite_residual), 0.5))

count_norm = LogNorm(vmin=max(vmin, 1), vmax=vmax)
residual_norm = SymLogNorm(linthresh=linthresh, vmin=-rlim, vmax=rlim)

PANELS = [
    ('Data',             air.data,         dict(norm=count_norm, cmap='viridis')),
    ('Supported model',  result.model,     dict(norm=count_norm, cmap='viridis')),
    ('Flags',            result.mask,      dict(cmap='gray_r', vmin=0, vmax=1)),
    ('Residual',         residual,         dict(norm=residual_norm, cmap='RdBu_r')),
    ('Flagged residual', flagged_residual, dict(norm=residual_norm, cmap='RdBu_r')),
]

# Colourbars go underneath rather than beside: an interior colourbar splits the
# five panels into two blocks and breaks the eye's left-to-right read.
fig, axes = plt.subplots(1, len(PANELS), figsize=(4.4 * len(PANELS), 8.0),
                         sharex=True, sharey=True, constrained_layout=True)
images = []
for axis, (title, array, kw) in zip(axes, PANELS):
    # White marks cells with no value at all: unsupported model, excluded data.
    images.append(axis.imshow(array, aspect='auto', extent=extent, **kw))
    axis.set_title(title)
    axis.set_xlabel('Frequency [MHz]')
    axis.tick_params(direction='out', length=4)
axes[0].set_ylabel('Minutes from selection start')

fig.colorbar(images[0], ax=axes[:2], orientation='horizontal',
             label='Correlator counts', shrink=0.8, aspect=40, pad=0.02)
fig.colorbar(images[3], ax=axes[3:], orientation='horizontal',
             label='Data − model [counts]', shrink=0.8, aspect=40, pad=0.02)
fig.suptitle('Supported-DPSS RFI flagger — %s, %s, %d files, %.0f min'
             % (CAMPAIGN.name, AIR_ANTENNA, len(approved_files), minutes[-1]),
             fontsize=19, y=1.05)
save(fig, 'rfi_waterfalls')
plt.show()

## 5. Band summary

Two panels suitable for a slide on their own: the time-averaged data and model
spectra with the retained-residual scatter beneath them, and the fraction of
integrations excluded as a function of frequency with the reported bands marked.
Averages are taken over unflagged cells only, so a fully-flagged channel is a
gap rather than a biased point.


In [ ]:
retained = np.where(result.mask, np.nan, air.data)
with np.errstate(invalid='ignore'):
    data_spec = np.nanmean(retained, axis=0)
    model_spec = np.nanmean(np.where(result.mask, np.nan, result.model), axis=0)
    resid_rms = np.sqrt(np.nanmean(flagged_residual ** 2, axis=0))
flag_frac = result.mask.mean(axis=0)

fig, (ax_spec, ax_flag) = plt.subplots(
    2, 1, figsize=(14, 9), sharex=True,
    gridspec_kw=dict(height_ratios=[2.2, 1]), constrained_layout=True)

# The reported bands are contiguous, so shade alternate ones: a span on every
# band merges into a single grey block that marks nothing.
for number, (name, (lo, hi)) in enumerate(REPORT_BANDS.items()):
    for axis in (ax_spec, ax_flag):
        if number % 2 == 0:
            axis.axvspan(lo, hi, color='0.92', zorder=0)
        axis.axvline(lo, color='0.75', lw=0.8, zorder=0)
    ax_spec.annotate(name.split(' ')[-1].strip('()') or name,
                     xy=(0.5 * (lo + hi), 1.005), xycoords=('data', 'axes fraction'),
                     ha='center', va='bottom', fontsize=11, color='0.35', rotation=90)

ax_spec.semilogy(freqs, data_spec, color='0.2', label='Data (unflagged mean)')
ax_spec.semilogy(freqs, model_spec, color='tab:orange', ls='--',
                 label='Supported background model')
ax_spec.semilogy(freqs, resid_rms, color='tab:blue', lw=1.6,
                 label='Residual RMS after flagging')
ax_spec.set_ylabel('Correlator counts')
ax_spec.legend(loc='lower left', ncol=1)

ax_flag.fill_between(freqs, flag_frac, color='tab:red', alpha=0.30, lw=0)
ax_flag.plot(freqs, flag_frac, color='tab:red', lw=1.6)
ax_flag.set_ylim(0, 1.02)
ax_flag.set_ylabel('Excluded fraction')
ax_flag.set_xlabel('Frequency [MHz]')
ax_flag.set_xlim(freqs[0], freqs[-1])

fig.suptitle('Time-averaged spectrum and excluded fraction — %s, %.0f min'
             % (AIR_ANTENNA, minutes[-1]), fontsize=19, y=1.07)
save(fig, 'rfi_band_summary')
plt.show()

## 6. Residual against the radiometer limit

If the background model removed everything real, the flagged residual would be
radiometer noise. It is not: there is structure at a few hundred ns of delay,
which at this site is canyon reflection rather than sky or interference.

The test scales the autocorrelation by the radiometer equation to get the
expected per-cell noise, fits a DPSS component over the reflection delay window
by inverse-variance weighted least squares (flagged cells carry zero weight, so
nothing is in-painted before fitting), and asks how close the remainder comes to
that noise.

**Open item:** the comparison is not yet properly calibrated for the degrees of
freedom the filtering removes. Projecting out N modes from M channels lowers the
residual RMS by ~sqrt(1 − N/M) even for pure noise, and that naive factor assumes
orthonormal modes and uniform weights — neither holds here, so it over-corrects.
The true ratio lies between the last two numbers printed below.


In [ ]:
from hera_filters.dspec import dpss_operator

# Delay range to remove, in ns. The canyon round-trip structure sits here.
DELAY_LO_NS, DELAY_HI_NS = 200.0, 500.0
EIGENVAL_CUTOFF = 1e-12
RIDGE = 1e-6          # fraction of mean(diag) added to stabilise the solve

freqs_hz = air.freqs_mhz * 1e6
dt = air.meta['integration_time'][0]
chan_bw_hz = 250e6 / air.meta['nchan'][0]

# Radiometer equation on an autocorrelation: sigma = 2 <P> / sqrt(2 B tau).
# The supported background model stands in for <P>, so this is the noise the
# residual would have if the model had removed everything real.
noise_sigma = 2 * result.model / np.sqrt(2 * chan_bw_hz * dt)

# Cells that carry information: kept by the flagger and backed by a model.
good = ~result.mask & np.isfinite(residual) & np.isfinite(noise_sigma)
print('usable cells: %.1f%% of the band' % (100 * good.mean()))

# DPSS basis spanning +-[DELAY_LO, DELAY_HI] ns: two windows, centred at the
# midpoint of the range with half-width half its span.
centre = 0.5 * (DELAY_LO_NS + DELAY_HI_NS) * 1e-9
halfwidth = 0.5 * (DELAY_HI_NS - DELAY_LO_NS) * 1e-9
A = dpss_operator(freqs_hz, filter_centers=[centre, -centre],
                  filter_half_widths=[halfwidth, halfwidth],
                  eigenval_cutoff=[EIGENVAL_CUTOFF, EIGENVAL_CUTOFF])[0]
n_chan, n_modes = A.shape

# Inverse-variance weighted least squares, one solve per integration, with the
# flagged cells carrying zero weight. A plain projection (A.T @ x) is NOT
# equivalent: these DPSS windows are not orthonormal over a gapped band, and
# projecting instead of solving spreads the fitted component across all delays.
# Weighting also removes any need to in-paint before fitting.
Ah = A.conj().T
weights = np.where(good, 1.0 / np.maximum(noise_sigma, 1e-9) ** 2, 0.0)
weight_scale = np.median(weights[weights > 0])
reflection_model = np.zeros_like(residual)
fitted_rows = 0
for row in range(residual.shape[0]):
    w = weights[row] / weight_scale
    if (w > 0).sum() < 4 * n_modes:      # too little support to constrain the fit
        continue
    AtW = Ah * w
    normal = AtW @ A
    normal.flat[::n_modes + 1] += RIDGE * np.trace(normal).real / n_modes
    reflection_model[row] = np.real(A @ np.linalg.solve(normal, AtW @ np.where(good, residual, 0.0)[row]))
    fitted_rows += 1

filtered = residual - reflection_model

print('delay window     : %.0f-%.0f ns (both signs)' % (DELAY_LO_NS, DELAY_HI_NS))
print('DPSS basis       : %d modes over %d channels' % (n_modes, n_chan))
print('integrations fit : %d of %d' % (fitted_rows, residual.shape[0]))
print('integration time : %.3f s   channel bw: %.1f kHz' % (dt, chan_bw_hz / 1e3))

# Removing N modes from M channels lowers the residual RMS by ~sqrt(1 - N/M)
# even for pure noise. Quote the ratio both ways.
dof_factor = np.sqrt(1.0 - n_modes / n_chan)
band = (result.freqs_mhz >= 50) & (result.freqs_mhz < 88)
keep = good & band
ratio_raw = np.sqrt(np.nanmean((residual[keep] / noise_sigma[keep]) ** 2))
ratio_filt = np.sqrt(np.nanmean((filtered[keep] / noise_sigma[keep]) ** 2))
print('\nover 50-88 MHz, %d usable cells:' % keep.sum())
print('  residual / expected noise          %.3f' % ratio_raw)
print('  after removing the delay window    %.3f' % ratio_filt)
print('  same, corrected for %d lost DOF   %.3f' % (n_modes, ratio_filt / dof_factor))
print('\nOPEN: the DOF correction is the naive sqrt(1 - N/M), which assumes the')
print('removed modes are orthonormal and uniformly weighted. Neither holds over a')
print('gapped, inverse-variance-weighted band, so it over-corrects: the true')
print('answer lies between the last two numbers.')

### Where the residual sits relative to the noise

In [ ]:
fig, (ax_spec, ax_ratio) = plt.subplots(
    2, 1, figsize=(14, 9), sharex=True,
    gridspec_kw=dict(height_ratios=[2.0, 1]), constrained_layout=True)

with np.errstate(invalid='ignore'):
    resid_rms = np.sqrt(np.nanmean(np.where(good, residual, np.nan) ** 2, axis=0))
    filt_rms = np.sqrt(np.nanmean(np.where(good, filtered, np.nan) ** 2, axis=0))
    noise_mean = np.nanmean(np.where(good, noise_sigma, np.nan), axis=0)

ax_spec.semilogy(freqs, noise_mean, color='0.2', lw=2.6,
                 label=r'Radiometer noise  $2\langle P\rangle/\sqrt{2B\tau}$')
ax_spec.semilogy(freqs, resid_rms, color='tab:blue',
                 label='Residual RMS (background model removed)')
ax_spec.semilogy(freqs, filt_rms, color='tab:green',
                 label='After removing %.0f–%.0f ns delay structure'
                       % (DELAY_LO_NS, DELAY_HI_NS))
ax_spec.set_ylabel('Correlator counts')
ax_spec.legend(loc='upper right')
ax_spec.set_title('Residual against the radiometer limit')

ax_ratio.axhline(1.0, color='0.2', lw=2.2)
ax_ratio.annotate('radiometer limit', xy=(freqs[-1], 1.0), xytext=(-6, 6),
                  textcoords='offset points', ha='right', fontsize=12, color='0.2')
ax_ratio.semilogy(freqs, resid_rms / noise_mean, color='tab:blue',
                  label='residual')
ax_ratio.semilogy(freqs, filt_rms / noise_mean, color='tab:green',
                  label='delay-filtered')
ax_ratio.semilogy(freqs, filt_rms / noise_mean / dof_factor, color='tab:green',
                  ls=':', label='delay-filtered, naive DOF correction')
ax_ratio.set_ylabel('Ratio to noise')
ax_ratio.set_xlabel('Frequency [MHz]')
ax_ratio.legend(loc='upper right', ncol=3)
ax_ratio.set_xlim(freqs[0], freqs[-1])
# Shade the band the printed ratios are quoted over.
for axis in (ax_spec, ax_ratio):
    axis.axvspan(50, 88, color='0.92', zorder=0)

save(fig, 'rfi_residual_vs_radiometer')
plt.show()

### The delay structure being removed

Left: the mean delay spectrum over 50-88 MHz, with the radiometer-noise floor for
reference and the fitted window shaded. Right: the same transform per
integration. Rows less than half-unflagged over this band are dropped.


In [ ]:
# Delay transform over the band the ratios are quoted on. Two reasons not to
# use the full 35-235 MHz: a third of it is flagged, so the transform would be
# dominated by whatever fills those gaps, and the sharp band edges ring across
# every delay. A Blackman-Harris taper suppresses what remains.
DELAY_BAND_MHZ = (50.0, 88.0)
sub = (result.freqs_mhz >= DELAY_BAND_MHZ[0]) & (result.freqs_mhz < DELAY_BAND_MHZ[1])
taper = np.blackman(sub.sum())
taper /= np.sqrt(np.mean(taper ** 2))

# In-paint at the expected noise level so the transform sees a gap-free band.
# Display only: the fit above is weighted and never sees an in-painted value.
rng = np.random.default_rng(0)
filler = rng.normal(size=residual.shape) * np.nan_to_num(noise_sigma)
inpainted = np.nan_to_num(np.where(good, residual, filler), nan=0.0)

# Rows with enough real data to be worth transforming.
usable_rows = good[:, sub].mean(axis=1) > 0.5
delays_ns = np.fft.fftshift(np.fft.fftfreq(sub.sum(), chan_bw_hz)) * 1e9


def delay_spectrum(array):
    return np.abs(np.fft.fftshift(
        np.fft.fft(array[usable_rows][:, sub] * taper, axis=1), axes=1))


ds_resid = delay_spectrum(inpainted)
ds_model = delay_spectrum(reflection_model)
# Subtract the fitted component only where there was real data. Taking it off
# the in-painted cells too would subtract a model from an independent noise
# draw, which *adds* variance and lifts this curve outside the fitted window.
ds_filt = delay_spectrum(np.where(good, inpainted - reflection_model, inpainted))
ds_noise = delay_spectrum(filler)

fig, (ax_mean, ax_fall) = plt.subplots(
    1, 2, figsize=(17, 7.5), constrained_layout=True,
    gridspec_kw=dict(width_ratios=[1.15, 1]))

ax_mean.semilogy(delays_ns, ds_noise.mean(axis=0), color='0.2', lw=2.4,
                 label='Radiometer noise (same in-paint draw)')
ax_mean.semilogy(delays_ns, ds_resid.mean(axis=0), color='tab:blue',
                 label='Residual')
ax_mean.semilogy(delays_ns, ds_model.mean(axis=0), color='tab:orange',
                 label='Fitted %.0f–%.0f ns component' % (DELAY_LO_NS, DELAY_HI_NS))
ax_mean.semilogy(delays_ns, ds_filt.mean(axis=0), color='tab:green',
                 label='Residual after removal')
for sign in (-1, 1):
    ax_mean.axvspan(sign * DELAY_LO_NS, sign * DELAY_HI_NS, color='0.91', zorder=0)
ax_mean.set_xlim(-1200, 1200)
ax_mean.set_xlabel('Delay [ns]')
ax_mean.set_ylabel('|delay transform|, mean over integrations')
ax_mean.legend(loc='lower center', fontsize=12, ncol=2)
# The fitted component falls many decades outside its window; without a floor
# it sets the y-range and squashes everything the figure is about.
_floor = ds_noise.mean(axis=0).min()
ax_mean.set_ylim(_floor / 5, max(ds_resid.mean(axis=0).max(),
                                 ds_model.mean(axis=0).max()) * 3)
ax_mean.set_title('Delay spectrum over %.0f–%.0f MHz' % DELAY_BAND_MHZ)

extent = [delays_ns[0], delays_ns[-1], ds_resid.shape[0], 0]
vmax = np.percentile(ds_resid, 99.5)
image = ax_fall.imshow(ds_resid, aspect='auto', extent=extent, cmap='plasma',
                       norm=LogNorm(vmin=vmax / 300, vmax=vmax))
for sign in (-1, 1):
    for edge in (DELAY_LO_NS, DELAY_HI_NS):
        ax_fall.axvline(sign * edge, color='w', lw=1.0, ls='--', alpha=0.8)
ax_fall.set_xlim(-1200, 1200)
ax_fall.set_xlabel('Delay [ns]')
ax_fall.set_ylabel('Integration (usable rows only)')
ax_fall.set_title('Residual delay spectrum per integration')
fig.colorbar(image, ax=ax_fall, label='|delay transform|', shrink=0.85, pad=0.02)

save(fig, 'rfi_delay_transform')
plt.show()

## Provenance and caveats

- Input time is `time_best` from `MetadataIndex`; physical antennas are resolved by the hashed campaign policy, and the resulting auto/cross keys and matched rules are shown above.
- The tested background-model domain is 35–235 MHz. Outside it, the public model is unsupported and the mask is set.
- The 300 ns correction has constant amplitude over this selected window. It can represent stable 35–88 MHz spectral structure, including stable interference; recovered pixels should not be interpreted as clean sky from the model alone.
- The correction is tapered at 35 and 88 MHz. The combined support diagnostic deliberately rejects edge predictions that become too weakly constrained.
- Fine-grained FM recovery is not being developed further in this beta. FM cells are treated by the same support and detector rules as the accepted approach.
- Band and comb triggers act on groups, then flag the participating region/tooth cells in unison for that integration. Point and cross-change tests remain cellwise.
- A segment that loses enough fit support is fully flagged with `unsupported_background` and a NaN model. The segment diagnostics above report this instead of assuming a successful solver history.
- This notebook does not write products. Use `eigsep-rfi-v3-beta` on complete files after reviewing this result.

## Decision requested

Review whether this updated `v3-beta` result is suitable for broader multi-file testing and product generation. Approval should specify the file range and whether overwriting any existing beta payload is allowed.

**STOPPED AT REVIEW GATE — awaiting Aaron's approval.**

In [ ]:
import uvtools.plot
import hera_filters
gnd = selection.load_bundle(antenna=GROUND_ANTENNA, band_mhz=config.band_mhz,
                            missing='raise', resolution_policy=resolution_policy)

In [ ]:
freq = air.freqs_mhz
d, m, r = air.data, result.model, residual
wgts = 1 - result.mask
B = 250e6 / air.meta['nchan'][0]
dt = air.meta['integration_time'][0]
noise_lev = np.nan_to_num(m, 0) / np.sqrt(B * dt)
inpaint = np.where(wgts > 0, d, np.nan_to_num(m, 0) + noise_lev * np.random.normal(size=m.shape))
A = hera_filters.dspec.dpss_operator(air.freqs_mhz * 1e6, filter_centers=[0], filter_half_widths=[40e-9], eigenval_cutoff=[1e-12])[0].real
F = hera_filters.dspec.fit_solution_matrix(np.ones_like(air.freqs_mhz), A)
inpaint_sm = (A @ (F @ inpaint.T)).T

In [ ]:
#plt.figure()
#uvtools.plot.waterfall(inpaint - inpaint_sm, mode='real', mx=1e5, drng=2e5, cmap='bwr')
plt.figure()
#plt.plot(freq, d[0])
#plt.plot(freq, m[0])
#plt.plot(r[0] * wgts[0], '.')
plt.plot(freq, inpaint[0] - m[0])
plt.plot(freq, inpaint[0] - inpaint_sm[0])